In [1]:
import numpy
from alternet.annotation import *
from alternet.data_preprocessing import standardize_dataframe
import numpy
import os
import os.path as op






In [2]:


data_path = "/data/bionets/og86asub/alternet-project/alternet/data"
results_path = "/data/bionets/og86asub/alternet-project/alternet/results-2.0"

# Reference files
appris_path = "appris_data.appris.txt"
digger_path = "digger_data.csv"
biomart_path = "biomart.txt"
tf_list_path = "allTFs_hg38.txt"
sf_list_path = "splicefactors.csv"

# Expression data
gtex_transcript_tpm_path = "GTEx_Analysis_v10_RSEMv1.3.3_transcripts_tpm.txt"
gtex_sample_attributes_path = "GTEx_Analysis_v10_Annotations_SampleAttributesDS.txt"

# Tissue to analyze
TISSUE = "Bladder"
CONDITION = TISSUE

# Number of GRNBoost2 runs
N_RUNS = 10

os.makedirs(results_path, exist_ok=True)



In [3]:
biomart = pd.read_csv(op.join(data_path, biomart_path), sep='\t')
tx2gene = dict(zip(biomart['Transcript stable ID'], biomart['Gene stable ID']))
gene2tx = biomart.groupby('Gene stable ID')['Transcript stable ID'].apply(set).to_dict()
appris_df = pd.read_csv(op.join(data_path,appris_path), sep='\t')
digger_df = pd.read_csv(op.join(data_path,digger_path), low_memory=False)
# Load and map TF list
tf_list_raw = pd.read_csv(op.join(data_path,tf_list_path), sep='\t', header=None)
tf_list = map_tf_ids(tf_list_raw, biomart)

In [4]:


VARIANCE_PERCENTILE = 0.7  # Keep top 30%

In [5]:
# Load and map SF list
sf_list_raw = pd.read_csv(op.join(data_path, sf_list_path), header=0, sep = ',')
sf_list = map_sf_ids(sf_list_raw.loc[:, ['Splicing_Factor']], biomart)
# Combine TF and SF lists
regulator_list = combine_tf_sf_lists(tf_list, sf_list)
tx_to_regtype = dict(zip(regulator_list['Transcript stable ID'], regulator_list['Regulator_type']))
gene_to_regtype = regulator_list.groupby('Gene stable ID')['Regulator_type'].first().to_dict()


In [6]:


# Create mappings
transcript_mapper = create_transcript_mapping(biomart)
print(f"Transcript-to-gene mappings: {len(transcript_mapper)}")

# Annotation databases
tf_database = create_transcipt_annotation_database(
    tf_list=tf_list, appris_df=appris_df, digger=digger_df
)
regulator_database = create_transcipt_annotation_database(
    tf_list=regulator_list, appris_df=appris_df, digger=digger_df
)
print(f"TF annotation database: {len(tf_database)} entries")
print(f"Regulator annotation database: {len(regulator_database)} entries")



Transcript-to-gene mappings: 278220
TF annotation database: 16298 entries
Regulator annotation database: 18958 entries


In [7]:
from alternet.data_preprocessing import *
from alternet.gtex_dataloader import *

In [8]:
gtex_data_dir = '/data/bionets/datasets/hackathon/data/GTEX'
params = {'sample_attributes': op.join(gtex_data_dir, 'GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt'), 'tissue': 'Liver', 'transcript_data':op.join(gtex_data_dir, 'GTEx_Analysis_2017-06-05_v8_RSEMv1.3.0_transcript_tpm.gct')}
tissue_ids = retrieve_GTEX_tissue_sampleids(params['sample_attributes'], tissue=params['tissue'])
transcript_data = read_GTEX_transcript_expression(params['transcript_data'], tissue_ids)
transcript_data = clean_GTEX_tissue_transcript_counts(transcript_data, biomart)
transcript_data = variance_filtering(transcript_data)


Retrieving tissue sample IDs
Reading Transcript expression data
Cleaning up counts


In [9]:

# sample_cols = [c for c in transcript_data.columns if c not in ['transcript_id', 'gene_id']]
# gene_data = transcript_data.groupby('gene_id')[sample_cols].sum().reset_index()



In [10]:

# # Create expression matrices (samples × features)
# gene_data = gene_data.set_index('gene_id')[sample_cols]
# transcript_data = transcript_data.set_index(['transcript_id', 'gene_id'])[sample_cols]



In [11]:
transcript_data

,transcript_id,gene_id,GTEX-11DXY-0526-SM-5EGGQ,GTEX-11DXZ-0126-SM-5EGGY,GTEX-11EQ9-0526-SM-5A5JZ,GTEX-11GSP-0626-SM-5986T,GTEX-11NUK-1226-SM-5P9GM,GTEX-11NV4-1326-SM-5HL6V,GTEX-11OF3-0726-SM-5BC4Z,GTEX-11TT1-1726-SM-5EQLJ,...,GTEX-ZF29-2026-SM-DNZYW,GTEX-ZF2S-3026-SM-4WWCH,GTEX-ZPU1-0826-SM-57WG2,GTEX-ZTPG-1426-SM-51MT3,GTEX-ZVP2-0626-SM-51MSO,GTEX-ZVT3-1626-SM-5GU66,GTEX-ZVT4-0626-SM-5E45T,GTEX-ZYT6-0626-SM-5E45V,GTEX-ZYY3-0626-SM-5NQ6W,GTEX-ZZPU-0426-SM-5GZYH
28,ENST00000374004,ENSG00000000938,0.14,0.03,1.67,0.56,0.59,1.29,0.27,1.51,...,0.98,0.86,0.56,10.85,3.22,1.52,2.45,1.76,2.06,1.63
29,ENST00000374005,ENSG00000000938,0.36,1.27,1.32,0.00,0.79,0.21,1.21,3.79,...,3.16,1.72,1.83,1.88,0.42,0.38,0.76,0.17,0.78,1.32
34,ENST00000359637,ENSG00000000971,57.68,0.00,0.00,50.96,41.76,50.15,33.72,0.00,...,0.70,0.59,0.40,26.68,35.69,31.23,39.19,75.34,52.81,0.00
35,ENST00000367429,ENSG00000000971,129.00,414.80,315.60,188.10,140.90,185.30,205.80,98.39,...,277.80,476.60,533.30,164.20,248.90,245.30,127.50,181.10,166.80,318.20
36,ENST00000466229,ENSG00000000971,28.10,31.16,27.73,61.59,48.71,46.35,86.55,6.27,...,11.40,18.87,46.23,21.18,21.82,89.37,39.80,26.14,42.09,27.06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198036,ENST00000241356,ENSG00000282608,0.03,1.52,1.71,0.29,0.10,0.06,0.33,0.63,...,1.35,1.02,0.15,1.30,0.74,0.19,0.12,0.11,0.92,1.11
198166,ENST00000635200,ENSG00000282988,0.25,0.00,0.00,0.18,0.64,0.36,0.13,2.45,...,0.28,0.15,0.15,2.06,0.18,0.79,0.59,0.20,1.79,0.21
198257,ENST00000418776,ENSG00000283149,6.86,0.75,8.58,12.81,5.16,10.87,6.79,43.94,...,8.85,5.26,19.73,12.39,8.51,10.48,11.56,6.99,10.78,15.09
198298,ENST00000636204,ENSG00000283189,3.98,2.36,2.09,4.31,3.76,3.96,3.57,0.70,...,0.84,1.53,0.51,1.43,0.24,3.32,1.28,2.68,2.85,2.16


In [ ]:
# gene_data_scaled = standardize_dataframe(gene_data).T
# transcript_data_scaled = standardize_dataframe(transcript_data).T
#transcript_data_scaled, gene_data_scaled, transcript_data = remove_problematic_transcripts(transcript_data_scaled, gene_data_scaled, transcript_data)

In [17]:
from alternet import postprocessing
from alternet.edge_categorization import *

In [18]:
transcript_data

,transcript_id,gene_id,GTEX-11DXY-0526-SM-5EGGQ,GTEX-11DXZ-0126-SM-5EGGY,GTEX-11EQ9-0526-SM-5A5JZ,GTEX-11GSP-0626-SM-5986T,GTEX-11NUK-1226-SM-5P9GM,GTEX-11NV4-1326-SM-5HL6V,GTEX-11OF3-0726-SM-5BC4Z,GTEX-11TT1-1726-SM-5EQLJ,...,GTEX-ZF29-2026-SM-DNZYW,GTEX-ZF2S-3026-SM-4WWCH,GTEX-ZPU1-0826-SM-57WG2,GTEX-ZTPG-1426-SM-51MT3,GTEX-ZVP2-0626-SM-51MSO,GTEX-ZVT3-1626-SM-5GU66,GTEX-ZVT4-0626-SM-5E45T,GTEX-ZYT6-0626-SM-5E45V,GTEX-ZYY3-0626-SM-5NQ6W,GTEX-ZZPU-0426-SM-5GZYH
28,ENST00000374004,ENSG00000000938,0.14,0.03,1.67,0.56,0.59,1.29,0.27,1.51,...,0.98,0.86,0.56,10.85,3.22,1.52,2.45,1.76,2.06,1.63
29,ENST00000374005,ENSG00000000938,0.36,1.27,1.32,0.00,0.79,0.21,1.21,3.79,...,3.16,1.72,1.83,1.88,0.42,0.38,0.76,0.17,0.78,1.32
34,ENST00000359637,ENSG00000000971,57.68,0.00,0.00,50.96,41.76,50.15,33.72,0.00,...,0.70,0.59,0.40,26.68,35.69,31.23,39.19,75.34,52.81,0.00
35,ENST00000367429,ENSG00000000971,129.00,414.80,315.60,188.10,140.90,185.30,205.80,98.39,...,277.80,476.60,533.30,164.20,248.90,245.30,127.50,181.10,166.80,318.20
36,ENST00000466229,ENSG00000000971,28.10,31.16,27.73,61.59,48.71,46.35,86.55,6.27,...,11.40,18.87,46.23,21.18,21.82,89.37,39.80,26.14,42.09,27.06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198036,ENST00000241356,ENSG00000282608,0.03,1.52,1.71,0.29,0.10,0.06,0.33,0.63,...,1.35,1.02,0.15,1.30,0.74,0.19,0.12,0.11,0.92,1.11
198166,ENST00000635200,ENSG00000282988,0.25,0.00,0.00,0.18,0.64,0.36,0.13,2.45,...,0.28,0.15,0.15,2.06,0.18,0.79,0.59,0.20,1.79,0.21
198257,ENST00000418776,ENSG00000283149,6.86,0.75,8.58,12.81,5.16,10.87,6.79,43.94,...,8.85,5.26,19.73,12.39,8.51,10.48,11.56,6.99,10.78,15.09
198298,ENST00000636204,ENSG00000283189,3.98,2.36,2.09,4.31,3.76,3.96,3.57,0.70,...,0.84,1.53,0.51,1.43,0.24,3.32,1.28,2.68,2.85,2.16


In [22]:
canonical_grn = pd.read_csv(op.join(results_path, f"{CONDITION}_canonical_raw.tsv"), sep='\t')

as_source_grn = pd.read_csv(op.join(results_path, f"{CONDITION}_as_aware_source_raw.tsv"), sep='\t')

fully_as_grn= pd.read_csv(op.join(results_path, f"{CONDITION}_fully_as_aware_raw.tsv"), sep='\t')


In [23]:
# Filtering Parameters
MIN_FREQUENCY = 10
IMPORTANCE_PERCENTILE = 0.7  # Keep top 30%


# Set C and Set D: PSI/Usage thresholds

DOM_MIN = 0.5
DOM_EQ_MIN = 0.7
GENE_TPM_MIN = 1.0,

# Set C specific
MIN_ISOFORMS_FOR_SPLICING = 2
TOP_M_EXPRESSED = 3



In [24]:
fully_as_grn = fully_as_grn.rename(columns={'source': 'source_transcript', 'target': 'target_transcript'})
fully_as_grn['source_gene'] = fully_as_grn['source_transcript'].map(tx2gene)
fully_as_grn['target_gene'] = fully_as_grn['target_transcript'].map(tx2gene)
fully_as_grn['reg_type'] = fully_as_grn['source_transcript'].map(tx_to_regtype)
as_source_grn = as_source_grn.rename(columns={'source': 'source_transcript', 'target': 'target_gene'})
as_source_grn['source_gene'] = as_source_grn['source_transcript'].map(tx2gene)
as_source_grn['reg_type'] = as_source_grn['source_transcript'].map(tx_to_regtype)
canonical_grn = canonical_grn.rename(columns={'source': 'source_gene', 'target': 'target_gene'})
canonical_grn['reg_type'] = canonical_grn['source_gene'].map(gene_to_regtype)

In [25]:
from alternet.alternet import *

In [28]:
alternet_obj21 = Alternet(canonical_grn, as_source_grn, fully_as_grn, transcript_data, regulator_list,  'gene_id', 'transcript_id')

Computing set A

Set A: 454,849 rows

Category distribution:
  source_gene_specific            257,769 ( 56.7%)
  source_isoform_specific         102,998 ( 22.6%)
  source_ambiguous                 50,358 ( 11.1%)
  source_equivalent                43,724 (  9.6%)
Set D diambiguation
Corrected version
Computing set D
Computing set B

Final Set B: 492,991 rows

Category distribution:
  target_isoform_specific         183,947 ( 37.3%)
  target_gene_specific            182,064 ( 36.9%)
  target_equivalent                69,012 ( 14.0%)
  target_ambiguous                 57,968 ( 11.8%)

By regulator type:
  n_target_tx == 0 (gene_specific, no Net3 edges): 137,631
  n_target_tx == 1 (resolved): 294,707
  n_target_tx >= 2 (multi): 60,653
  target_tx_dominant non-empty: 355,360
       source_transcript      source_gene      target_gene  \
0        ENST00000198939  ENSG00000085872  ENSG00000005882   
1        ENST00000198939  ENSG00000085872  ENSG00000008517   
2        ENST00000198939  ENSG0